# Part 3 — Constrained Decoding

Constrained (Hebrew-only) decoding for `Qwen/Qwen2.5-7B-Instruct` and `mistralai/Mistral-7B-Instruct-v0.3`.

**Run first (from terminal):**
```
python code/part3/run_part3.py
```
That script produces (in the workspace root):
- `hebrew_allowed_tokens_qwen.json`
- `hebrew_allowed_tokens_mistral.json`
- `decoding_outputs.jsonl`

This notebook loads those files and displays the results.

## 3.1 — Hebrew Token Identification

In [1]:
import json
import pandas as pd

TOKEN_FILES = {
    "Qwen/Qwen2.5-7B-Instruct":           "../../hebrew_allowed_tokens_qwen.json",
    "mistralai/Mistral-7B-Instruct-v0.3": "../../hebrew_allowed_tokens_mistral.json",
}
token_data = {}
for model_id, path in TOKEN_FILES.items():
    with open(path, encoding="utf-8") as f:
        token_data[model_id] = json.load(f)

rows = []
for model_id, d in token_data.items():
    rows.append({
        "model_id":      model_id,
        "allowed_count": len(d["allowed_token_ids"]),
    })

df_tokens = pd.DataFrame(rows)
df_tokens

,model_id,allowed_count
0,Qwen/Qwen2.5-7B-Instruct,13234
1,mistralai/Mistral-7B-Instruct-v0.3,1912


In [2]:
import random
from transformers import AutoTokenizer

SAMPLE_SIZE = 1000

for model_id, d in token_data.items():
    print(f"\n{'='*70}")
    print(f"Model: {model_id}")
    print(f"Total allowed tokens: {len(d['allowed_token_ids']):,}")
    print(f"{'='*70}")

    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)

    sampled_ids = random.sample(d["allowed_token_ids"], min(SAMPLE_SIZE, len(d["allowed_token_ids"])))
    sampled_ids.sort()

    decoded = [tok.decode([tid]) for tid in sampled_ids]

    # Display as a wrapped grid (20 per row) for readability
    col_width = 12
    per_row = 20
    for i in range(0, len(decoded), per_row):
        row = decoded[i : i + per_row]
        print("  " + "".join(repr(t).ljust(col_width) for t in row))

    del tok

/home/kapachy/.venvs/llms_ass2_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Model: Qwen/Qwen2.5-7B-Instruct
Total allowed tokens: 13,234


  "'"         '0'         '�'         '�'         '�'         '�'         '�'         '�'         '\x08'      '\x14'      '\x17'      '\x1a'      '�'         ' ('        "('"        "',"        '__'        ' @'        '"\n'       ');\r\n'    
  ' |'        '                                ''":'        '\n\n\n\n'  ' },\n'     ',"'        ' ""'       '}\r\n'     ' (('       '\\"'       "';\n\n"    '())\n'     '};\n'      '}\r\n\r\n' '\t   '     ' ${'       "'))"       ' /**'      '="<?'      ').\n'      
  '\t\r\n'    '"/>\n'     '                                                                ''、'         '})'        ':@"'       ']:'        ']\r\n'     '\t\t\r\n'  ' "-'       ' .\n\n'    ' \'"'      ' ©'        '”\n\n'     '                              ''________________'"'];"       '<='        '�'         ' "</'      
  ')}'        "=='"       '}",'       '[:,'       "'\r\n"     ' �'        ' },\r\n'   ')"'        "':'"       '�'         '([]'       '("@'       '/$'        '=-'       

## 3.2 — Inference Results

In [3]:
records = []
with open("../../decoding_outputs.jsonl", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Loaded {len(records)} records  ({df['model'].nunique()} models × {df['prompt'].nunique()} queries)")
df[["model", "prompt"]]

Loaded 20 records  (2 models × 10 queries)


,model,prompt
0,Qwen/Qwen2.5-7B-Instruct,Explain why the sky looks blue during the day.
1,Qwen/Qwen2.5-7B-Instruct,Give two advantages and two disadvantages of p...
2,Qwen/Qwen2.5-7B-Instruct,Write a short email asking a professor for an ...
3,Qwen/Qwen2.5-7B-Instruct,Describe how to make a simple omelette.
4,Qwen/Qwen2.5-7B-Instruct,What is the difference between supervised and ...
5,Qwen/Qwen2.5-7B-Instruct,Summarize the story of Cinderella in three sen...
6,Qwen/Qwen2.5-7B-Instruct,Suggest three ways to reduce smartphone distra...
7,Qwen/Qwen2.5-7B-Instruct,Explain what happens when water boils.
8,Qwen/Qwen2.5-7B-Instruct,Give a polite refusal to an invitation to a pa...
9,Qwen/Qwen2.5-7B-Instruct,"Turn the idea ""practice makes progress"" into a..."


In [4]:
pd.set_option("display.max_colwidth", 120)

for i, query in enumerate(df["prompt"].unique(), 1):
    print(f"\n{'='*70}")
    print(f"Query {i}: {query}")
    print('='*70)
    for _, row in df[df["prompt"] == query].iterrows():
        short = row["model"].split("/")[-1]
        print(f"\n  [{short}] Unconstrained:")
        print(f"    {row['unconstrained_output'][:400]}")
        print(f"\n  [{short}] Constrained (Hebrew-only):")
        print(f"    {row['constrained_output'][:400]}")


Query 1: Explain why the sky looks blue during the day.

  [Qwen2.5-7B-Instruct] Unconstrained:
    The sky appears blue during the day due to a phenomenon called Rayleigh scattering. This process is named after Lord Rayleigh, who first described it in the late 19th century.

Here's how it works:

1. **Sunlight Composition**: Sunlight is composed of white light, which is actually a mixture of all colors of the visible spectrum (red, orange, yellow, green, blue, indigo, and violet). Each color co

  [Qwen2.5-7B-Instruct] Constrained (Hebrew-only):
    !';
### **[1. **[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**[**

  [Mistral-7B-Instruct-v0.3] Unconstrain